# Task 1, Step 4: P3 Nonuniform Wanda Pruning

The weight-ranking rule stays Wanda (per-output-row); only *where* pruning is aggressive changes.

**Allocation unit:** a full Transformer block (all 7 eligible projections share s_l). All 28 blocks have the same
number of eligible weights, so mean_l s_l = S gives the same zero count as P1/P2; integer rounding is fixed by
`exact_zero_targets`, which makes the total equal round(S * N_eligible) exactly. Bounds 0.10 <= s_l <= 0.85.

**Two sensitivity measures (both kept, chosen between on the development windows only):**
* **A. OWL outlier ratio** (Yin et al. 2024): fraction of a block's Wanda scores above M=5 x block mean, from
  the dense model. Rule: higher outlier ratio -> lower sparsity, s_l = S - lambda * centred normalised LOD.
* **B. Isolated-removal Delta NLL** (ShortGPT / BRECQ-style block probe): prune only block l with Wanda at
  s in {0.10, 0.30, 0.50, 0.70, 0.85} (others dense) and measure the NLL increase on the first 32 calibration
  windows. Rule: exact DP minimising the summed (additive) Delta NLL at mean S on a 0.025 grid, then shrink
  towards uniform with beta.

**Selection protocol, fixed before any test evaluation:** candidates {OWL lambda in {0.04, 0.08, 0.12},
DP beta in {0.5, 1.0}} x S in {0.5, 0.75}; prune each with the full sequential Wanda pipeline and pick the lowest
NLL on the 32 disjoint development windows. Uniform (P2) is also scored on dev for reference. Only the selected
allocation is evaluated on WikiText-2 / C4 / HellaSwag.

Limitation: the isolated probe ignores interactions between pruned blocks (additivity assumption).

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task1_compression")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task1", need_public=True, need_prefix_banks=False)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import save_json, load_json
from shared.src.plotting import save_figure
from task1_compression.src.runner import setup_task1
from task1_compression.src import prune, allocation
from task1_compression.src.evaluate import nll_on_sequences
from task1_compression.src.model_utils import layers_of, proj_module, ELIGIBLE

T = setup_task1(ctx)
model = T.model
norms = torch.load(ctx.checkpoints / "dense_input_norms.pt")   # from notebook 2
PROBE = T.calib[:32]                                           # fixed probe subset (same IDs for every probe)
TARGETS = (0.50, 0.75)

## Sensitivity A: OWL layer outlier ratio

In [ ]:
T.snap.restore(model)
t0 = time.perf_counter()
lod = allocation.owl_outlier_ratios(model, norms, M=5.0)
owl_time = time.perf_counter() - t0
plt.figure(figsize=(9, 3)); plt.bar(range(28), lod); plt.xlabel("block"); plt.ylabel("outlier ratio (M=5)")
plt.title("OWL layer outlier distribution (dense model, calibration activations)")
save_figure(ctx.figures, "p3_owl_outlier_ratio")

## Sensitivity B: isolated block-removal Delta NLL on calibration windows

In [ ]:
probe_path = ctx.results / "p3_block_probe.json"
grid = allocation.probe_grid()
if probe_path.exists():
    probe = load_json(probe_path)
else:
    T.snap.restore(model)
    t0 = time.perf_counter()
    base_nll = nll_on_sequences(model, PROBE)
    curves = np.zeros((28, len(grid)))
    for li, layer in enumerate(layers_of(model)):
        backup = {p: proj_module(layer, p).weight.data.clone() for p in ELIGIBLE}
        for j, s in enumerate(grid):
            prune.prune_block_oneshot(layer, norms[li], s, "wanda")
            curves[li, j] = nll_on_sequences(model, PROBE) - base_nll
            for p in ELIGIBLE:
                proj_module(layer, p).weight.data.copy_(backup[p])
        print(f"block {li}: dNLL@0.5={curves[li, 2]:.4f}  @0.85={curves[li, -1]:.4f}", flush=True)
    probe = dict(grid=grid, base_nll=base_nll, curves=curves.tolist(), probe_windows=32, time_s=time.perf_counter() - t0,
                 owl_lod=lod.tolist(), owl_time_s=owl_time)
    save_json(probe_path, probe)
curves = np.array(probe["curves"])
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.heatmap(curves, ax=ax[0], cmap="rocket_r", xticklabels=grid, cbar_kws=dict(label="Delta NLL (nats/token)"))
ax[0].set_xlabel("isolated block sparsity"); ax[0].set_ylabel("block"); ax[0].set_title("Isolated Wanda removal sensitivity")
ax[1].scatter(lod, curves[:, 2]); ax[1].set_xlabel("OWL outlier ratio"); ax[1].set_ylabel("Delta NLL @ s=0.5")
ax[1].set_title(f"Spearman = {pd.Series(lod).corr(pd.Series(curves[:, 2]), method='spearman'):.2f}")
save_figure(ctx.figures, "p3_sensitivity_probe")

## Candidate allocations and development-set selection

In [ ]:
cand_path = ctx.results / "p3_candidates.json"
cands = load_json(cand_path) if cand_path.exists() else {}

def dev_score(alloc, S, tag):
    if tag in cands:
        return cands[tag]["dev_nll"]
    T.snap.restore(model)
    z = prune.exact_zero_targets(model, {i: float(a) for i, a in enumerate(alloc)}, S)
    t0 = time.perf_counter()
    r = prune.prune_sequential(model, T.calib, "wanda", z, gram=False)
    d = nll_on_sequences(model, T.dev)
    cands[tag] = dict(S=S, alloc=[float(a) for a in alloc], dev_nll=d, zeros=r["zeros"], time_s=time.perf_counter() - t0)
    save_json(cand_path, cands)
    print(f"{tag}: dev NLL {d:.4f}", flush=True)
    return d

for S in TARGETS:
    dev_score(np.full(28, S), S, f"S{S}_uniform")
    for lam in (0.04, 0.08, 0.12):
        dev_score(allocation.owl_allocation(lod, S, lam), S, f"S{S}_owl_lam{lam}")
    dp = allocation.dp_allocation(curves, grid, S)
    for beta in (0.5, 1.0):
        dev_score(allocation.shrink(dp, S, beta), S, f"S{S}_dp_beta{beta}")

cdf = pd.DataFrame(cands).T
cdf[["S", "dev_nll", "zeros", "time_s"]]

In [ ]:
selected = {}
for S in TARGETS:
    sub = {k: v for k, v in cands.items() if v["S"] == S and "uniform" not in k}
    best = min(sub, key=lambda k: sub[k]["dev_nll"])
    selected[S] = best
    print(f"S={S}: selected {best} (dev NLL {sub[best]['dev_nll']:.4f}; uniform {cands[f'S{S}_uniform']['dev_nll']:.4f})")
save_json(ctx.results / "p3_selection.json", dict(rule="lowest NLL on 32 C4-train development windows among nonuniform candidates",
                                                  selected={str(k): v for k, v in selected.items()}))

## Final P3 runs (held-out evaluation only for the selected allocations)

In [ ]:
for S in TARGETS:
    c = cands[selected[S]]
    def apply(m, c=c, S=S):
        zt = prune.exact_zero_targets(m, {i: a for i, a in enumerate(c["alloc"])}, S)
        return prune.prune_sequential(m, T.calib, "wanda", zt, gram=True)
    r = T.runner.run(f"p3_s{int(S*100)}", apply_fn=apply,
                     meta=dict(method="wanda nonuniform", unit="block", candidate=selected[S], allocation=c["alloc"]))

## Matched-budget check, sparsity maps, paired comparisons vs P2

In [ ]:
from task1_compression.src.analysis import load_condition, compare_table, summary_table
for S in TARGETS:
    a = load_condition(ctx.results, f"p3_s{int(S*100)}")["compression"]["stats"]["zeros"]
    b = load_condition(ctx.results, f"p2_wanda_s{int(S*100)}")["compression"]["stats"]["zeros"]
    print(f"S={S}: P3 zeros {a:,} vs P2 zeros {b:,} -> equal: {a == b}")
    assert a == b

fig, ax = plt.subplots(figsize=(10, 3.5))
for S in TARGETS:
    ax.plot(cands[selected[S]]["alloc"], "o-", label=f"S={S} ({selected[S]})")
    ax.axhline(S, ls="--", lw=0.8, color="gray")
ax.set_xlabel("block"); ax.set_ylabel("block sparsity s_l"); ax.legend(); ax.set_ylim(0.05, 0.9)
save_figure(ctx.figures, "p3_final_sparsity_map")

In [ ]:
compare_table(ctx.results, [("p3_s50", "p2_wanda_s50"), ("p3_s75", "p2_wanda_s75"), ("p3_s50", "fp16"), ("p3_s75", "fp16")])

In [ ]:
summary_table(ctx.results).loc[["fp16", "p2_wanda_s50", "p3_s50", "p2_wanda_s75", "p3_s75"]]